# Análise Exploratória dos Dados
aqui vou auditar os dados e documentar os achados, fazendo análise de Qualidade

In [0]:
#carregando meu stack básico
from pyspark.sql import functions as F
CATALOGO, SCHEMA = "workspace", "default"
fato = spark.table(f"{CATALOGO}.{SCHEMA}.gold_fato_scada")

### 1. Completude
Quantos nulos tenho por coluna?

In [0]:
total = fato.count()

#contar nulos por cada coluna
colunas_medida = [
    "wind_speed_m_s",
    "power_kw",
    "rotor_speed_rpm",
    "generator_rpm",
    "blade_pitch_a_deg",
    "front_bearing_temp_c",
    "rear_bearing_temp_c",
    "gear_oil_temp_c"
] #nao estou levando em conta colunas derivadas tipo min, max e medio

for coluna in colunas_medida:
    nulos = fato.filter(F.col(coluna).isNull()).count()
    pct = round(100 * nulos / total, 2)
    print(f"{coluna:35s} nulos: {nulos}  {pct}%")

wind_speed_m_s                      nulos: 11373  3.61%
power_kw                            nulos: 11002  3.49%
rotor_speed_rpm                     nulos: 11002  3.49%
generator_rpm                       nulos: 11002  3.49%
blade_pitch_a_deg                   nulos: 11002  3.49%
front_bearing_temp_c                nulos: 11002  3.49%
rear_bearing_temp_c                 nulos: 11002  3.49%
gear_oil_temp_c                     nulos: 11002  3.49%


coerente pois, com uma parada, é esperado que todos os outros sensores acompanhe a pausa/falha e por isso retornem nulos também. destaque para a coluna de velocidade do vento, que possuem nulos ligeiramente maiores, que pode ser por falha pontual do anemometro em outras ocorrencias.

### 2. Unicidade da chave
o grão do fato é "uma linha por turbina × instante". Se existir duas linhas pra turbina 1 às 09:20, o modelo está furado

In [0]:
#agrupando pela chave de propósito
duplicatas = (fato
              .groupBy("turbina_id", "tempo_id")
              .count()
              .filter(F.col("count") > 1)
             )

qtd = duplicatas.count()
print(f"qtd duplicatas: {qtd}")

#mostrando alguns exemplos, caso haja
if qtd > 0:
    display(duplicatas.orderBy(F.desc("count").limit(10)))
else:
    print("nenhuma duplicata encontrada")

qtd duplicatas: 0
nenhuma duplicata encontrada


ok, chave única e a granularidade está ok também

## 3. Curadoria dos dados
Inspeciono domínio, faixas, valores, o que faz sentido com a realidade ou não

In [0]:
# faixas fisicamente plausíveis pra cada medida
# (turbina Senvion MM92: 2050 kW nominal, corte ~3 m/s, ~25 m/s de parada)
checagens = [
    # coluna,            mínimo, máximo, motivo
    ("power_kw",           -50,   2200,  "potência: aceita pequeno negativo (consumo parada), teto acima do nominal"),
    ("wind_speed_m_s",       0,     30,  "vento: nunca negativo; 30 m/s já é teto de acordo com manual"),
    ("rotor_speed_rpm",      0,     16,  "rotor: gira devagar, ~16 rpm nominal"),
    ("generator_rpm",        0,  1820,  "gerador: milhares de rpm"),
    ("blade_pitch_a_deg",  0,    360,  "ajuste da pá: ângulo em graus"),
    ("front_bearing_temp_c",-40,    150,  "temperatura de mancal em operação"),
    ("gear_oil_temp_c",    -40,    150,  "temperatura normal do óleo"),
    ("nacelle_ambient_temp_c",-40,   45,  "ambiente: faixa climática do Reino Unido com folga"),
]

for coluna, minimo, maximo, motivo in checagens:
    fora = fato.filter(
        (F.col(coluna) < minimo) | (F.col(coluna) > maximo)
    ).count()
    print(f"{coluna:28s} fora de [{minimo}, {maximo}]: {fora:6d}   # {motivo}")

power_kw                     fora de [-50, 2200]:      0   # potência: aceita pequeno negativo (consumo parada), teto acima do nominal
wind_speed_m_s               fora de [0, 30]:      0   # vento: nunca negativo; 30 m/s já é teto de acordo com manual
rotor_speed_rpm              fora de [0, 16]:      0   # rotor: gira devagar, ~16 rpm nominal
generator_rpm                fora de [0, 1820]:   2031   # gerador: milhares de rpm
blade_pitch_a_deg            fora de [0, 360]:      0   # ajuste da pá: ângulo em graus
front_bearing_temp_c         fora de [-40, 150]:      0   # temperatura de mancal em operação
gear_oil_temp_c              fora de [-40, 150]:      0   # temperatura normal do óleo
nacelle_ambient_temp_c       fora de [-40, 45]:      0   # ambiente: faixa climática do Reino Unido com folga


o rpm do gerador tem dados estranhos. preciso avaliar mais a fundo.

In [0]:
display(fato.select(
    F.min("generator_rpm").alias("gen_rpm_min"),
    F.max("generator_rpm").alias("gen_rpm_max"),
))

gen_rpm_min,gen_rpm_max
-589.2462158203125,1817.098876953125


In [0]:
fato = spark.table(f"{CATALOGO}.{SCHEMA}.gold_fato_scada")

negativos = fato.filter(F.col("generator_rpm") < 0)

# 1) confirmar que os 2031 fora de faixa são exatamente os negativos
print("generator_rpm negativos:", negativos.count(), "(esperado: 2031)")

# 2) esses negativos coincidem com turbina parada/sem gerar?
print("  ...com power_kw baixo (<10) ou nulo:",
      negativos.filter((F.col("power_kw") < 10) | F.col("power_kw").isNull()).count())

# 3) e coincidem com o status de parada (status_id = 1)?
print("  ...com status_id = 1 (Stop):",
      negativos.filter(F.col("status_id") == 1).count())

# 3) e coincidem com o status de parada (status_id = 1)?
print("  ...com rotor_speed_rpm = 0 (Stop):",
      negativos.filter(F.col("rotor_speed_rpm") < 0.5 ).count())

# 4) ver alguns pra sentir o padrão
display(negativos.select("generator_rpm", "rotor_speed_rpm", "power_kw", "wind_speed_m_s", "status_id")
    .orderBy("generator_rpm").limit(10))

generator_rpm negativos: 2031 (esperado: 2031)
  ...com power_kw baixo (<10) ou nulo: 2031
  ...com status_id = 1 (Stop): 421
  ...com rotor_speed_rpm = 0 (Stop): 2031


generator_rpm,rotor_speed_rpm,power_kw,wind_speed_m_s,status_id
-589.2462158203125,0.0,-5.623089790344238,8.409375190734863,1
-582.386474609375,0.0,-1.5151400566101074,0.7113746404647827,1
-580.2172241210938,0.0,-7.789100170135498,5.151374816894531,1
-580.038330078125,0.0,-0.864270031452179,10.57906723022461,1
-291.37469482421875,0.0,-2.7902050018310547,10.505812644958496,1
-291.14044189453125,0.0,-1.179034948348999,11.269514083862305,1
-148.8850860595703,0.0,-3.766510009765625,9.250614166259766,1
-147.6631622314453,0.0,-2.723517656326294,11.139656066894531,1
-118.20106506347656,0.0,-3.7473039627075195,1.4504249095916748,1
-99.30982208251953,0.0,-2.807988405227661,2.667250156402588,1


A auditoria de domínio sinalizou 2.031 leituras (0,6% do total) com generator_rpm negativo. Como rotação negativa é fisicamente impossível (o eixo não gira em sentido contrário), a hipótese inicial foi de ruído de sensor. A investigação confirmou a hipótese com três evidências independentes e convergentes:

100% coincidem com potência nula ou negativa (2.031 de 2.031) — a turbina não estava gerando;
100% coincidem com rotor praticamente parado (rotor_speed_rpm < 0,5 rpm) — não havia giro mecânico;
421 das 2.031 caem dentro de janelas de parada já identificadas na reconciliação (status_id = 1, Stop).

Diversas dessas leituras ocorreram com vento acima da velocidade de corte (8–11 m/s), o que descarta falta de vento como causa: a turbina estava parada por outro motivo, e o sensor de rotação, sem giro para medir, oscilou em torno de zero e ocasionalmente reportou valores negativos.

Decisão de tratamento: os valores negativos de generator_rpm são marcados como inválidos (null) na camada curada (gold_fato_scada_curado), preservando a tabela original intacta.

Observação para trabalhos futuros: apenas 421 dos ~2.031 registros de parada foram capturados pela reconciliação, que considerou somente eventos do tipo Stop. Os demais correspondem a paradas registradas sob outros tipos de evento (Warning, Communication) ou não formalizadas no log. Expandir a reconciliação a esses tipos aumentaria a cobertura das paradas identificadas.

In [0]:
fato_curado = (fato
    # gerador negativo é ruído de sensor (turbina parada) -> vira null
    .withColumn("generator_rpm",
        F.when(F.col("generator_rpm") < 0, None).otherwise(F.col("generator_rpm")))
    # rotor fora de [0, 16] (folga sobre os 15 rpm do manual) -> null
    .withColumn("rotor_speed_rpm",
        F.when((F.col("rotor_speed_rpm") < 0) | (F.col("rotor_speed_rpm") > 16), None)
         .otherwise(F.col("rotor_speed_rpm")))
    # pitch: convenção circular 0–360°, nada a limpar
)

(fato_curado.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.{SCHEMA}.gold_fato_scada_curado"))

print("gold_fato_scada_curado criado!")

# validação: os negativos do gerador devem ter sumido
curado = spark.table(f"{CATALOGO}.{SCHEMA}.gold_fato_scada_curado")
print("generator_rpm negativos restantes:",
      curado.filter(F.col("generator_rpm") < 0).count(), "(deve ser 0)")

gold_fato_scada_curado criado!
generator_rpm negativos restantes: 0 (deve ser 0)
